# CrewAI NFL Agents (ESPN)

This notebook sets up **one CrewAI agent per ESPN NFL API endpoint** and provides helper functions to run them.

Endpoints used:

- Scores: `http://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard`
- News: `http://site.api.espn.com/apis/site/v2/sports/football/nfl/news`
- All Teams: `http://site.api.espn.com/apis/site/v2/sports/football/nfl/teams`
- Specific Team: `http://site.api.espn.com/apis/site/v2/sports/football/nfl/teams/:team`

## 1) Install dependencies

In [ ]:
pip install crewai crewai-tools gradio requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2) Configure LLM environment (OpenAI or Azure OpenAI)

In [ ]:
import os

# === Choose your LLM backend ===
# Option A: OpenAI (uncomment and set your key)
# os.environ['OPENAI_API_KEY'] = 'YOUR_OPENAI_KEY'
# os.environ['OPENAI_MODEL'] = 'gpt-4o-mini'

# Option B: Azure OpenAI (uncomment and set these, and comment out OpenAI above)
# os.environ['AZURE_OPENAI_API_KEY'] = 'YOUR_AZURE_KEY'
# os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://YOUR_RESOURCE_NAME.openai.azure.com/'
# os.environ['AZURE_OPENAI_API_VERSION'] = '2024-08-01-preview'
# os.environ['AZURE_OPENAI_DEPLOYMENT_NAME'] = 'gpt-4o-mini'

# If you prefer a .env file in the same folder, uncomment:
# from dotenv import load_dotenv
# load_dotenv()  # Loads variables from .env into os.environ


## 3) Imports, constants, and HTTP helper

In [ ]:
import json
import time
from typing import Optional, Dict, Any, List
import requests
from crewai import Agent, Task, Crew, Process
from crewai_tools import tool
import gradio as gr
import re, json
from crewai import Crew, Process

In [ ]:
# ESPN endpoints
ESPN_BASE = 'http://site.api.espn.com/apis/site/v2/sports/football/nfl'
SCORES_URL = f'{ESPN_BASE}/scoreboard'
NEWS_URL = f'{ESPN_BASE}/news'
TEAMS_URL = f'{ESPN_BASE}/teams'

DEFAULT_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (CrewAI NFL Agents; +https://local-notebook)'}
REQUEST_TIMEOUT = 15


def http_get_json(url: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """GET JSON with retry/backoff and helpful errors for notebook use.

    Args:
        url: full URL
        params: optional query params
    Returns:
        dict parsed JSON
    Raises:
        RuntimeError on repeated failures
    """
    backoff = 1.0
    for attempt in range(4):
        try:
            resp = requests.get(url, params=params, headers=DEFAULT_HEADERS, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            return resp.json()
        except (requests.RequestException, ValueError) as e:
            if attempt < 3:
                time.sleep(backoff)
                backoff *= 2
            else:
                raise RuntimeError(f'Failed to GET {url} with params={params}: {e}') from e


## 4) Define one Tool per ESPN endpoint

In [ ]:
@tool('get_nfl_scores', return_direct=False)
def get_nfl_scores(dates: Optional[str] = None) -> str:
    """
    Fetch NFL scoreboard JSON.
    dates: YYYYMMDD (e.g., '20240908'). If None, ESPN returns 'today' (in season).
    Returns raw JSON (string).
    """
    params = {'dates': dates} if dates else None
    data = http_get_json(SCORES_URL, params=params)
    return json.dumps(data)

@tool('get_nfl_news', return_direct=False)
def get_nfl_news(limit: int = 20) -> str:
    """Fetch NFL news JSON (raw string). limit is best-effort.

    """
    params = {'limit': limit} if limit else None
    data = http_get_json(NEWS_URL, params=params)
    return json.dumps(data)

@tool('get_all_nfl_teams', return_direct=False)
def get_all_nfl_teams() -> str:
    """Fetch all NFL teams JSON (raw string).

    """
    data = http_get_json(TEAMS_URL)
    return json.dumps(data)

@tool('get_specific_nfl_team', return_direct=False)
def get_specific_nfl_team(team: str) -> str:
    """
    Fetch a specific team by id or slug (e.g., '24', 'ne', 'new-england-patriots').
    Returns raw JSON (string).
    """
    url = f'{TEAMS_URL}/{team}'
    data = http_get_json(url)
    return json.dumps(data)


## 5) (Optional) Simplifiers for clean, compact outputs

In [ ]:
def simplify_scoreboard(raw_json: Dict[str, Any]) -> List[Dict[str, Any]]:
    games = []
    for ev in raw_json.get('events', []):
        game_id = ev.get('id')
        competitions = ev.get('competitions', [])
        comp = competitions[0] if competitions else {}
        status_info = (comp.get('status') or {}).get('type', {})
        status_desc = status_info.get('description')
        start = status_info.get('shortDetail') or ev.get('date')

        competitors = comp.get('competitors', [])
        home = next((c for c in competitors if c.get('homeAway') == 'home'), None)
        away = next((c for c in competitors if c.get('homeAway') == 'away'), None)

        def team_info(c):
            if not c:
                return {}
            team = c.get('team', {})
            return {
                'id': team.get('id'),
                'abbreviation': team.get('abbreviation'),
                'displayName': team.get('displayName'),
                'score': c.get('score'),
                'winner': c.get('winner'),
            }

        games.append({
            'gameId': game_id,
            'status': status_desc,
            'start': start,
            'home': team_info(home),
            'away': team_info(away),
        })
    return games

def simplify_news(raw_json: Dict[str, Any], limit: int = 10) -> List[Dict[str, Any]]:
    items = []
    for a in raw_json.get('articles', [])[:limit]:
        items.append({
            'headline': a.get('headline'),
            'description': a.get('description'),
            'published': a.get('published'),
            'link': (a.get('links') or {}).get('web', {}).get('href'),
            'images': [i.get('url') for i in a.get('images', []) if i.get('url')],
            'byline': a.get('byline'),
            'source': a.get('source'),
        })
    return items

def simplify_teams(raw_json: Dict[str, Any]) -> List[Dict[str, Any]]:
    out = []
    for g in raw_json.get('sports', []):
        for l in g.get('leagues', []):
            for t in l.get('teams', []):
                info = t.get('team', {})
                out.append({
                    'id': info.get('id'),
                    'slug': info.get('slug'),
                    'abbreviation': info.get('abbreviation'),
                    'displayName': info.get('displayName'),
                    'shortDisplayName': info.get('shortDisplayName'),
                    'location': info.get('location'),
                    'name': info.get('name'),
                    'color': info.get('color'),
                    'logo': (info.get('logos') or [{}])[0].get('href') if info.get('logos') else None
                })
    return out


## 6) Define the agents (one per tool)

In [ ]:
scores_agent = Agent(
    name='Scores Agent',
    role='Fetches and summarizes NFL scoreboard data',
    goal=('Call get_nfl_scores and output a compact JSON of games.'),
    backstory=('An expert retriever that reads ESPN scoreboard and normalizes it.'),
    tools=[get_nfl_scores],
    verbose=True,
    allow_delegation=False,
)

news_agent = Agent(
    name='News Agent',
    role='Fetches latest NFL news',
    goal=('Call get_nfl_news and output an array of {headline, link, published}.'),
    backstory=('Surfaces timely NFL headlines with clean links.'),
    tools=[get_nfl_news],
    verbose=True,
    allow_delegation=False,
)

teams_agent = Agent(
    name='Teams Agent',
    role='Lists all NFL teams',
    goal=('Call get_all_nfl_teams and output compact team objects.'),
    backstory=('Keeps an updated roster of NFL clubs.'),
    tools=[get_all_nfl_teams],
    verbose=True,
    allow_delegation=False,
)

team_agent = Agent(
    name='Team Agent',
    role='Fetches details for a specific team by id or slug',
    goal=('Call get_specific_nfl_team and return key team fields.'),
    backstory=('Team detail specialist.'),
    tools=[get_specific_nfl_team],
    verbose=True,
    allow_delegation=False,
)


## 7) Define tasks for each agent

In [ ]:
scores_task = Task(
    description=(
        "Fetch NFL scoreboard for the provided 'dates' (YYYYMMDD) if given, else today. "
        "Call get_nfl_scores and return a JSON array: "
        "[{gameId, status, start, home:{id,abbreviation,displayName,score}, "
        "away:{id,abbreviation,displayName,score}}]"
    ),
    agent=scores_agent,
    expected_output='JSON array of simplified game objects.',
    output_json=True,
)

news_task = Task(
    description=(
        "Fetch latest NFL news (limit 10). Call get_nfl_news(limit=10). "
        "Return a JSON array [{headline, link, published}]."
    ),
    agent=news_agent,
    expected_output='JSON array of simplified news objects.',
    output_json=True,
)

teams_task = Task(
    description=(
        "Fetch all NFL teams and return a JSON array with fields: "
        "{id, displayName, abbreviation, slug}."
    ),
    agent=teams_agent,
    expected_output='JSON array of simplified team objects.',
    output_json=True,
)

team_task = Task(
    description=(
        "Given 'team' input (id or slug), call get_specific_nfl_team(team). "
        "Return a concise JSON with {id, slug, displayName, abbreviation, logo}."
    ),
    agent=team_agent,
    expected_output='JSON object of key team fields.',
    output_json=True,
)


## 8) Build the crew + helper functions to run from cells

In [ ]:
nfl_crew = Crew(
    agents=[scores_agent, news_agent, teams_agent, team_agent],
    tasks=[scores_task, news_task, teams_task, team_task],
    process=Process.sequential,
    verbose=True,
)

def run_scores(dates: Optional[str] = None):
    result = nfl_crew.kickoff(inputs={'dates': dates}, task=scores_task)
    try:
        return json.loads(result)
    except Exception:
        return [{'raw': result}]

def run_news(limit: int = 10):
    result = nfl_crew.kickoff(inputs={'limit': limit}, task=news_task)
    try:
        return json.loads(result)
    except Exception:
        return [{'raw': result}]

def run_teams():
    result = nfl_crew.kickoff(task=teams_task)
    try:
        return json.loads(result)
    except Exception:
        return [{'raw': result}]

def run_team(team: str):
    result = nfl_crew.kickoff(inputs={'team': team}, task=team_task)
    try:
        return json.loads(result)
    except Exception:
        return {'raw': result}


## 9) Use the agents interactively

In [ ]:
# Scores (today)
scores_today = run_scores()
scores_today[:3]  # preview first 3 games if available


In [ ]:
# Scores for a specific date (YYYYMMDD)
# Example date; adjust to an in-season date for more results.
scores_specific = run_scores('20240908')
len(scores_specific), scores_specific[:2]


In [ ]:
# News (top 5)
news_top5 = run_news(limit=5)
news_top5


In [ ]:
# All teams
teams = run_teams()
len(teams), teams[:5]


In [ ]:
# Specific team by slug or id
patriots = run_team('new-england-patriots')  # or 'ne' or numeric id
patriots


## 10) Gradio

In [ ]:

DATE_YMD_RE = re.compile(r"\b(20\d{2})(\d{2})(\d{2})\b")
LIMIT_RE = re.compile(r"\b(?:top|limit)\s*(\d{1,3})", re.IGNORECASE)

def extract_dates(question: str):
    m = DATE_YMD_RE.search(question or "")
    return f"{m.group(1)}{m.group(2)}{m.group(3)}" if m else None

def extract_limit(question: str, default=5):
    m = LIMIT_RE.search(question or "")
    if not m:
        return default
    try:
        v = int(m.group(1))
        return max(1, min(v, 50))
    except:
        return default

def run_scores_via_crew(dates=None):
    crew = Crew(agents=[scores_agent], tasks=[scores_task], process=Process.sequential, verbose=False)
    res = crew.kickoff(inputs={"dates": dates})
    try:
        return json.loads(res)
    except:
        return [{"raw": res}]

def run_news_via_crew(limit=5):
    crew = Crew(agents=[news_agent], tasks=[news_task], process=Process.sequential, verbose=False)
    res = crew.kickoff(inputs={"limit": limit})
    try:
        return json.loads(res)
    except:
        return [{"raw": res}]

def run_teams_via_crew():
    crew = Crew(agents=[teams_agent], tasks=[teams_task], process=Process.sequential, verbose=False)
    res = crew.kickoff()
    try:
        return json.loads(res)
    except:
        return [{"raw": res}]

def run_team_via_crew(team: str):
    crew = Crew(agents=[team_agent], tasks=[team_task], process=Process.sequential, verbose=False)
    res = crew.kickoff(inputs={"team": team})
    try:
        return json.loads(res)
    except:
        return {"raw": res}

def run_query(user_question: str) -> str:
    if not user_question or not user_question.strip():
        return "Please enter a question."

    q = user_question.lower()

    if "team" in q or "patriots" in q or "cowboys" in q:
        toks = re.findall(r"[a-z0-9-]+", q)
        candidate = next((t for t in toks if t not in ("team","teams") and len(t) >= 2), None)
        if not candidate:
            return ("I couldn't determine the team. Try including the team slug or abbreviation, "
                    "e.g., `ne` or `new-england-patriots`.")
        data = run_team_via_crew(candidate)
        return f"```json\n{json.dumps(data, indent=2)}\n```"

    if "score" in q or "game" in q or "schedule" in q:
        dates = extract_dates(q)
        games = run_scores_via_crew(dates=dates)
        return f"```json\n{json.dumps(games, indent=2)}\n```"

    if "news" in q or "headline" in q or "article" in q:
        limit = extract_limit(q, 5)
        items = run_news_via_crew(limit=limit)
        return f"```json\n{json.dumps(items, indent=2)}\n```"

    if "teams" in q or "all teams" in q or "list teams" in q or "franchises" in q:
        teams = run_teams_via_crew()
        return f"```json\n{json.dumps(teams, indent=2)}\n```"

    items = run_news_via_crew(limit=5)
    return f"```json\n{json.dumps(items, indent=2)}\n```"

EXAMPLE_QUESTIONS = [
    ["What are today’s NFL scores?"],
    ["Top 5 NFL news headlines"],
    ["List all NFL teams"],
    ["Show team info for new-england-patriots"],
    ["Scores for 20240908"],
]

import gradio as gr

with gr.Blocks(title="NFL CrewAI Agents") as demo:
    gr.Markdown("# 🏈 NFL CrewAI Agents")
    gr.Markdown("Ask about NFL scores, news, teams, or a specific team (use slug or abbreviation).")

    with gr.Row():
        question_box = gr.Textbox(
            label="Your Question",
            placeholder="e.g. What are today's NFL scores?",
            lines=2,
            scale=4,
        )
        submit_btn = gr.Button("Ask", variant="primary", scale=1)

    answer_box = gr.Markdown()

    gr.Examples(examples=EXAMPLE_QUESTIONS, inputs=question_box)

    submit_btn.click(fn=run_query, inputs=question_box, outputs=answer_box)
    question_box.submit(fn=run_query, inputs=question_box, outputs=answer_box)

demo.launch()